# Mixed Precision Accumulation

This notebook contains the code for Problem `mixed_precision_accumulation` from the assignment handout.

## Problem Statement

Run the following code and comment on the accuracy of the results.

Deliverable: A 2-3 sentence response.

In [1]:
import torch

print("Torch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

Torch version: 2.6.0+cu124
CUDA available: True


In [2]:
# FP32 accumulator, FP32 increment
s = torch.tensor(0, dtype=torch.float32)
for i in range(1000):
    s += torch.tensor(0.01, dtype=torch.float32)
print("FP32 + FP32:", s)

FP32 + FP32: tensor(10.0001)


In [3]:
# FP16 accumulator, FP16 increment
s = torch.tensor(0, dtype=torch.float16)
for i in range(1000):
    s += torch.tensor(0.01, dtype=torch.float16)
print("FP16 + FP16:", s)

FP16 + FP16: tensor(9.9531, dtype=torch.float16)


In [4]:
# FP32 accumulator, FP16 increment created inline
s = torch.tensor(0, dtype=torch.float32)
for i in range(1000):
    s += torch.tensor(0.01, dtype=torch.float16)
print("FP32 + inline FP16:", s)

FP32 + inline FP16: tensor(10.0021)


In [5]:
# FP32 accumulator, FP16 increment explicitly cast back to FP32
s = torch.tensor(0, dtype=torch.float32)
for i in range(1000):
    x = torch.tensor(0.01, dtype=torch.float16)
    s += x.type(torch.float32)
print("FP32 + casted FP16->FP32:", s)

FP32 + casted FP16->FP32: tensor(10.0021)


In [6]:
expected = 10.0

results = {}

s = torch.tensor(0, dtype=torch.float32)
for i in range(1000):
    s += torch.tensor(0.01, dtype=torch.float32)
results["fp32_acc_fp32_inc"] = float(s)

s = torch.tensor(0, dtype=torch.float16)
for i in range(1000):
    s += torch.tensor(0.01, dtype=torch.float16)
results["fp16_acc_fp16_inc"] = float(s)

s = torch.tensor(0, dtype=torch.float32)
for i in range(1000):
    s += torch.tensor(0.01, dtype=torch.float16)
results["fp32_acc_inline_fp16_inc"] = float(s)

s = torch.tensor(0, dtype=torch.float32)
for i in range(1000):
    x = torch.tensor(0.01, dtype=torch.float16)
    s += x.type(torch.float32)
results["fp32_acc_casted_fp16_inc"] = float(s)

for name, value in results.items():
    print(f"{name}: {value:.10f} | abs error = {abs(value - expected):.10f}")

fp32_acc_fp32_inc: 10.0001335144 | abs error = 0.0001335144
fp16_acc_fp16_inc: 9.9531250000 | abs error = 0.0468750000
fp32_acc_inline_fp16_inc: 10.0021362305 | abs error = 0.0021362305
fp32_acc_casted_fp16_inc: 10.0021362305 | abs error = 0.0021362305


## Writeup Notes

Use this cell to draft the 2-3 sentence response for the assignment.

- The FP16 accumulator should show the largest error because repeated additions are rounded in low precision.
- Keeping the accumulator in FP32 is much more accurate, even if the increment itself started as FP16.
- Explicitly casting the FP16 value back to FP32 before accumulation makes the high-precision accumulation behavior more obvious.